In [1]:
# data_generator.py
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import random
import time
def get_spark(app_name="SparkOptDemo", configs=None):
    """Create a SparkSession with optional config overrides."""
    builder = SparkSession.builder.appName(app_name).master("local[*]")
    if configs:
        for k, v in configs.items():
            builder = builder.config(k, v)
    return builder.getOrCreate()

def generate_large_table(spark, num_rows=10_000_000, num_partitions=200):
    """Simulate a protocol-events-style table (large fact table)."""
    return (
        spark.range(0, num_rows, numPartitions=num_partitions)
        .withColumn("account_number", (F.col("id") % 500_000).cast("string"))
        .withColumn("event_date", F.date_add(F.lit("2025-01-01"), (F.col("id") % 90).cast("int")))
        .withColumn("session_duration", (F.rand() * 3600).cast("int"))
        .withColumn("download_bytes", (F.rand() * 1_000_000).cast("long"))
        .withColumn("upload_bytes", (F.rand() * 500_000).cast("long"))
        .withColumn("category", F.concat(F.lit("cat_"), (F.col("id") % 50).cast("string")))
    )

def generate_small_lookup(spark, num_rows=5_000):
    """Simulate a small dimension / lookup table (e.g., URL categories)."""
    return (
        spark.range(0, num_rows)
        .withColumn("category", F.concat(F.lit("cat_"), F.col("id").cast("string")))
        .withColumn("category_name", F.concat(F.lit("Category "), F.col("id").cast("string")))
        .withColumn("priority", (F.rand() * 10).cast("int"))
    )

def generate_skewed_table(spark, num_rows=10_000_000, hot_keys=5, hot_key_ratio=0.6):
    """
    Generate a table where a few 'hot' keys hold 60% of the data.
    This simulates real-world data skew (e.g., a few heavy-usage accounts).
    """
    hot_rows = int(num_rows * hot_key_ratio)
    normal_rows = num_rows - hot_rows

    hot_df = (
        spark.range(0, hot_rows)
        .withColumn("account_number", (F.col("id") % hot_keys).cast("string"))
        .withColumn("value", F.rand() * 1000)
    )
    normal_df = (
        spark.range(0, normal_rows)
        .withColumn("account_number", (F.col("id") % 100_000 + hot_keys).cast("string"))
        .withColumn("value", F.rand() * 1000)
    )
    return hot_df.union(normal_df)

In [2]:
spark=get_spark("Case01_BroadcastJoin")

In [3]:
large_df=generate_large_table(spark,num_rows=2000_000)
small_df=generate_small_lookup(spark,num_rows=5_000)
large_df.cache().count()
small_df.cache().count()

5000

In [5]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

start = time.time()

result_bad = (
    large_df
    .join(small_df, "category", "inner")
    .select(large_df["*"], small_df["priority"])
)

result_bad.write.mode("overwrite").parquet("./tmp/case01_bad")

print(f"❌ SortMergeJoin: {time.time() - start:.1f}s")

❌ SortMergeJoin: 1.8s


In [7]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "50MB")  # re-enable

start = time.time()
result_good = large_df.join(F.broadcast(small_df), "category", "inner").select(large_df["*"], small_df["priority"])
result_good.write.mode("overwrite").parquet("/tmp/case01_good")
print(f"✅ BroadcastJoin: {time.time() - start:.1f}s")

✅ BroadcastJoin: 1.8s


In [8]:
# Show the execution plans
print("\n--- BAD PLAN ---")
result_bad.explain(True)
print("\n--- GOOD PLAN ---")
result_good.explain(True)


--- BAD PLAN ---
== Parsed Logical Plan ==
'Project [ResolvedStar(id#0L, account_number#2, event_date#5, session_duration#9, download_bytes#14L, upload_bytes#20L, category#27), priority#44]
+- Project [category#27, id#0L, account_number#2, event_date#5, session_duration#9, download_bytes#14L, upload_bytes#20L, id#35L, category_name#40, priority#44]
   +- Join Inner, (category#27 = category#37)
      :- Project [id#0L, account_number#2, event_date#5, session_duration#9, download_bytes#14L, upload_bytes#20L, concat(cat_, cast((id#0L % cast(50 as bigint)) as string)) AS category#27]
      :  +- Project [id#0L, account_number#2, event_date#5, session_duration#9, download_bytes#14L, cast((rand(6778827972581455307) * cast(500000 as double)) as bigint) AS upload_bytes#20L]
      :     +- Project [id#0L, account_number#2, event_date#5, session_duration#9, cast((rand(8368322684046325083) * cast(1000000 as double)) as bigint) AS download_bytes#14L]
      :        +- Project [id#0L, account_numb

In [4]:
spark=get_spark("Case02_Bucketing",{
    "spark.sql.sources.bucketing.enabled": "true",
    "spark.sql.autoBroadcastJoinThreshold": "-1"
})

In [5]:
events_df = generate_large_table(spark, num_rows=1000_000)
sessions_df = (
    generate_large_table(spark, num_rows=5_000_000)
    .withColumnRenamed("session_duration", "total_session_time")
    .select("account_number", "total_session_time", "event_date")
)

In [6]:
sessions_df.show(3)

+--------------+------------------+----------+
|account_number|total_session_time|event_date|
+--------------+------------------+----------+
|             0|              2901|2025-01-01|
|             1|              2623|2025-01-02|
|             2|               811|2025-01-03|
+--------------+------------------+----------+
only showing top 3 rows



In [7]:
events_df.show(3)

+---+--------------+----------+----------------+--------------+------------+--------+
| id|account_number|event_date|session_duration|download_bytes|upload_bytes|category|
+---+--------------+----------+----------------+--------------+------------+--------+
|  0|             0|2025-01-01|            2101|        854629|      141060|   cat_0|
|  1|             1|2025-01-02|             170|        841670|       66952|   cat_1|
|  2|             2|2025-01-03|             318|        875264|      422608|   cat_2|
+---+--------------+----------+----------------+--------------+------------+--------+
only showing top 3 rows



In [8]:
# =============================================
# ❌ BAD: Join without bucketing (full shuffle)
# =============================================
events_df.write.mode("overwrite").parquet("./tmp/events_no_bucket")
sessions_df.write.mode("overwrite").parquet("./tmp/sessions_no_bucket")


In [9]:
events_nb = spark.read.parquet("./tmp/events_no_bucket")
sessions_nb = spark.read.parquet("./tmp/sessions_no_bucket")

In [13]:
start = time.time()
result_bad=events_nb.join(sessions_nb, "account_number", "inner").select(events_nb["*"],sessions_nb["total_session_time"])
result_bad.write.mode("overwrite").parquet("./tmp/case02_bad")
print(f"❌ Unbucketed join: {time.time() - start:.1f}s")

❌ Unbucketed join: 4.5s


In [14]:
# =============================================
# ✅ GOOD: Write bucketed tables then join
# =============================================
NUM_BUCKETS = 64
(events_df
 .write
 .mode("overwrite")
 .bucketBy(NUM_BUCKETS, "account_number")
 .sortBy("account_number")
 .saveAsTable("events_bucketed"))

(sessions_df
 .write
 .mode("overwrite")
 .bucketBy(NUM_BUCKETS, "account_number")
 .sortBy("account_number")
 .saveAsTable("sessions_bucketed"))






In [15]:
spark.conf.get("spark.sql.warehouse.dir")

'file:/home/jovyan/notebooks/spark-warehouse'

In [16]:
events_b = spark.table("events_bucketed")
sessions_b = spark.table("sessions_bucketed")

In [22]:
start = time.time()

result_good = (
    events_b
    .join(sessions_b, "account_number", "inner")
    .select(
        events_b["*"],
        sessions_b["total_session_time"]
    )
)

result_good.write.mode("overwrite").parquet("./tmp/case02_good")

print(f"✅ Bucketed join: {time.time() - start:.1f}s")

✅ Bucketed join: 8.0s
